# Project structure

What each folder in the repository is for. Every command runs from the repository root, and paths
in the code are relative to it.

```
ETAI-Project/
├── configs/       config overrides
├── data/raw/      input CSVs
├── docs/          notes on the data
├── notebooks/     exploration and experiments
├── src/           pipeline code
├── models/        saved models (generated)
├── results/       predictions and scores (generated)
├── main.py
├── config.yaml
└── requirements.txt
```

## Root files

| File | Purpose |
|---|---|
| `main.py` | Command-line entry point (`train`, `predict`). Reads the config and calls `src/`. No logic of its own. |
| `config.yaml` | Every setting that can change: paths, seed, features, models, CV. The single source of truth. |
| `requirements.txt` | Python packages with pinned versions. |
| `README.md` | How to set up the environment and run the pipeline on Mac and Windows. |

## `configs/`

Override files. Each one holds only the keys that differ from `config.yaml` and is merged on top of it.

- `dev.yaml`: a fast smoke run (fewer folds, smaller models). Writes to `models/dev` and `results/dev`.
- Future `exp_*.yaml` files: one experiment each.

## `data/raw/`

The four input files from the brief. Not committed to Git.

| File | Content |
|---|---|
| `train.csv` | 6,534 purchases with both targets |
| `test.csv` | 986 later purchases, no targets |
| `clientes.csv` | Optional customer table, joined on `customer_id` |
| `sample_submission.csv` | Kaggle submission format |

## `docs/`

Written notes that support the decisions in the code.

- `data_notes.md`: raw profile of the four files (shapes, missing values, categories, open questions).

## `notebooks/`

Exploration and experiments. Notebooks import from `src/` and never copy its code, so what we analyse
here is what the pipeline runs. Outputs are cleared before committing.

| Notebook | Content |
|---|---|
| `00_project_structure.ipynb` | This overview |
| `01_eda.ipynb` | Exploratory data analysis and cleaning decisions |

## `src/`

The pipeline code, one module per step.

| Module | Purpose |
|---|---|
| `data.py` | Load the CSVs, join `clientes.csv`, split into features, target and customer groups |
| `cleaning.py` | Fixed cleaning rules that keep every row; the same code runs on train, test and client input |
| `features.py` | Feature engineering, as a step of the model pipeline |
| `preprocessing.py` | Imputation, encoding and scaling |
| `selection.py` | Feature selection |
| `models.py` | Available models and the full pipeline: features, preprocessing, selection, model |
| `splitting.py` | Cross-validation splits and the locked holdout, grouped by customer |
| `evaluate.py` | Metrics and cross-validation |
| `tuning.py` | Hyperparameter tuning with Optuna |
| `train.py` | Train, evaluate and save a model with its metadata |
| `predict.py` | Batch prediction for `test.csv` and the Kaggle file |
| `client.py` | Prediction for a single new purchase |

Modules marked as TODO in the code are not implemented yet.

## `models/` and `results/`

Created when the pipeline runs. Not committed to Git.

- `models/`: the trained pipeline (`<task>.joblib`) and its metadata (`<task>_meta.json`: config,
  features, threshold, CV scores). `predict` and `client` only load these, they never retrain.
- `results/`: the Kaggle submission, the regression predictions and the evaluation scores.